In [23]:
import pandas as pd
import os

# 1. Caminho base onde os dados estão agrupados
pasta_base = 'regional'

# 2. Liste as subpastas (as cidades) dentro de 'regional'
cidades = [f for f in os.listdir(pasta_base) if os.path.isdir(os.path.join(pasta_base, f))]
print(f"Cidades reais encontradas dentro de 'regional': {cidades}\n")

# 3. Pegue a primeira cidade encontrada para espiar
primeira_cidade = cidades[0]
caminho_da_cidade = os.path.join(pasta_base, primeira_cidade)

# 4. Busque os arquivos .csv dentro dessa cidade
arquivos_semanas = [arq for arq in os.listdir(caminho_da_cidade) if arq.endswith('.csv')]
primeiro_arquivo = arquivos_semanas[0]

caminho_final_teste = os.path.join(caminho_da_cidade, primeiro_arquivo)
print(f"Espiando com sucesso o arquivo: '{primeiro_arquivo}'")
print(f"Localizado em: {caminho_final_teste}\n")

# 5. Leia o arquivo com o Pandas
df_espiada = pd.read_csv(caminho_final_teste)

# 6. O Raio-X das colunas
print("COLUNAS ENCONTRADAS NA TABELA:")
print(list(df_espiada.columns))
print("-" * 60)
print(f"Tamanho do arquivo espiado: {df_espiada.shape[0]} linhas e {df_espiada.shape[1]} colunas.")
print("-" * 60)

# Mostra o topo dos dados para vermos o que tem dentro
df_espiada.head(3)


Cidades reais encontradas dentro de 'regional': ['Belo Horizonte', 'Bel�m', 'Bras�lia', 'Campinas', 'Campo Grande', 'Cuiab�', 'Curitiba', 'Florian�polis', 'Fortaleza', 'Goi�nia', 'Manaus', 'Porto Alegre', 'Recife', 'Rio de Janeiro', 'Salvador', 'S�o Paulo', 'Uberl�ndia']

Espiando com sucesso o arquivo: 'Apr 1 - 7, 2022.csv'
Localizado em: regional\Belo Horizonte\Apr 1 - 7, 2022.csv

COLUNAS ENCONTRADAS NA TABELA:
['rank', 'uri', 'track_name', 'peak', 'prev', 'streak', 'artist_names']
------------------------------------------------------------
Tamanho do arquivo espiado: 100 linhas e 7 colunas.
------------------------------------------------------------


,rank,uri,track_name,peak,prev,streak,artist_names
0,1,spotify:track:3sd2p4kE7xQmFH3lPnFl6h,sentaDONA (Remix) s2,1,3,3,Davi Kneip
1,2,spotify:track:5zNxUkUW9WiV5oyg6gh2wB,DANÇARINA,1,2,8,PEDRO SAMPAIO
2,3,spotify:track:5719MEdRQcH4c3phXqX7WN,Mal Feito - Ao Vivo,2,4,12,Hugo & Guilherme


In [22]:
import pandas as pd
import os

pasta_base = 'regional'
todas_as_semanas = []

# 1. Pegue a lista de todas as cidades (pastas)
cidades = [f for f in os.listdir(pasta_base) if os.path.isdir(os.path.join(pasta_base, f))]

print("Iniciando a consolidação de todas as capitais... Aguarde um momento.")

# 2. Loop para entrar na pasta de cada cidade
for cidade in cidades:
    caminho_cidade = os.path.join(pasta_base, cidade)
    arquivos_csv = [arq for arq in os.listdir(caminho_cidade) if arq.endswith('.csv')]
    
    # 3. Loop para ler cada arquivo de semana dentro daquela cidade
    for arquivo in arquivos_csv:
        caminho_arquivo = os.path.join(caminho_cidade, arquivo)
        
        # Lemos o CSV daquela semana específica
        df_semana = pd.read_csv(caminho_arquivo)
        
        # Criamos as colunas identificadoras baseadas nas pastas e arquivos
        df_semana['cidade'] = cidade
        df_semana['semana'] = arquivo.replace('.csv', '') # Remove o '.csv' do nome da data
        
        # Guardamos essa semana na nossa lista
        todas_as_semanas.append(df_semana)

# 4. Juntar todas as tabelas em um único DataFrame gigante
df_garimpo = pd.concat(todas_as_semanas, ignore_index=True)

print("Consolidação concluída com sucesso!")
print(f"O nosso super DataFrame 'df_garimpo' foi criado com {df_garimpo.shape[0]} linhas!")
print("-" * 60)

# Mostrar uma amostra aleatória de 5 linhas para checar se a cidade e semana entraram certo
df_garimpo.sample(5)


Iniciando a consolidação de todas as capitais... Aguarde um momento.
Consolidação concluída com sucesso!
O nosso super DataFrame 'df_garimpo' foi criado com 201736 linhas!
------------------------------------------------------------


,rank,uri,track_name,peak,prev,streak,artist_names,cidade,semana
76549,45,spotify:track:7afaZ37wJJL6880z1NRgJS,Fim de Semana no Rio,15,45,16,Teto,Curitiba,"Jul 15 - 21, 2022"
78924,28,spotify:track:5yRgMxdDhdYlbUQNCOlW3o,Não Vou Te Bloquear - Ao vivo,26,26,7,Mc Don Juan,Curitiba,"Mar 4 - 10, 2022"
14613,35,spotify:track:7AwVSKaQxpidrtYBerkHKk,Pipoco,1,36,28,Ana Castela,Bel�m,"Dec 30 - Jan 5, 2022"
172668,93,spotify:track:7nsk0cwY5kfxBT21WEzmC0,Ele Não Te Merece,59,93,4,Wesley Safadão,Salvador,"Jun 24 - 30, 2022"
153540,40,spotify:track:7qoIhutxU269Zqo9PG5IOj,Poesia Acústica 13,40,—,1,Pineapple StormTv,Recife,"Sep 16 - 22, 2022"


In [21]:
# 1. Filtrar apenas as músicas que NUNCA chegaram ao topo absoluto (#1)
# O 'peak > 1' garante que tiramos todas as músicas que bateram o primeiro lugar
df_sem_topo = df_garimpo[df_garimpo['peak'] > 1]

# 2. Agrupar por música/artista e descobrimos qual foi o maior 'streak' (semanas seguidas) que ela alcançou
# Usar também o .max() no peak para saber qual foi a melhor colocação que ela conseguiu beliscar
resiliencia = df_sem_topo.groupby(['track_name', 'artist_names']).agg(
    max_streak=('streak', 'max'),
    melhor_pico=('peak', 'min') # 'min' porque no ranking, quanto menor o número, mais perto do topo (ex: 2 é melhor que 50)
).reset_index()

# 3. Ordenar do maior streak para o menor e pegamos os 10 maiores casos de resiliência nacional
top_resiliência = resiliencia.sort_values(by='max_streak', ascending=False).head(10)

print("OS 10 MAIORES CASOS DE RESILIÊNCIA REGIONAL (Sem nunca bater o #1):")
top_resiliência


OS 10 MAIORES CASOS DE RESILIÊNCIA REGIONAL (Sem nunca bater o #1):


,track_name,artist_names,max_streak,melhor_pico
1218,Poesia Acústica #6: Era uma Vez,Pineapple StormTv,118,37
1701,X1,MC Cabelinho,97,2
1622,Vampiro,Matuê,91,2
517,ESSÊNCIA DE CRIA,MC Cabelinho,88,15
696,Good Vibe,Filipe Ret,83,12
699,Gostosinha é Ela,Felipe Amorim,78,2
817,Licor 43,KayBlack,76,48
1654,Vizão de Cria 2,Dallass,76,17
138,As It Was,Harry Styles,74,8
763,Insônia - Ao Vivo,LUDMILLA,73,9


In [20]:
# 1. Filtrar os dados para separar o "Eixo (RJ-SP)" das "Capitais de Resistência" que queremos analisar
df_eixo = df_garimpo[df_garimpo['cidade'].isin(['São Paulo', 'Rio de Janeiro'])]
df_resistencia = df_garimpo[df_garimpo['cidade'].isin(['Recife', 'Manaus', 'Fortaleza', 'Salvador'])]

# 2. Descobrir qual foi a melhor posição que cada música alcançou no Eixo RJ-SP
pico_no_eixo = df_eixo.groupby(['track_name', 'artist_names'])['rank'].min().reset_index()
pico_no_eixo.rename(columns={'rank': 'melhor_rank_eixo'}, inplace=True)

# 3. Descobrir qual foi a melhor posição que cada música alcançou em Recife ou BH
pico_na_resistencia = df_resistencia.groupby(['track_name', 'artist_names', 'cidade'])['rank'].min().reset_index()
pico_na_resistencia.rename(columns={'rank': 'melhor_rank_local'}, inplace=True)

# 4. Juntar as duas informações usando a 'uri' ou o nome da música e artista
# Usamos 'how="left"' porque queremos manter todas as músicas que bombaram na resistência
analise_monopolio = pd.merge(pico_na_resistencia, pico_no_eixo, on=['track_name', 'artist_names'], how='left')

# Se a música nunca entrou no eixo, o Pandas vai colocar um valor vazio (NaN). Vamos preencher com 101 (fora do top 100)
analise_monopolio['melhor_rank_eixo'] = analise_monopolio['melhor_rank_eixo'].fillna(101)

# 5. O FILTRO CRÍTICO: Bombou no Top 10 local, mas ficou abaixo de #50 ou nem entrou no eixo RJ-SP
filtro_critico = (analise_monopolio['melhor_rank_local'] <= 10) & (analise_monopolio['melhor_rank_eixo'] > 50)
resultado_critico = analise_monopolio[filtro_critico]

# 6. Mostrar os 10 casos mais berrantes de sucesso independente e puramente regional
resultado_critico.sort_values(by='melhor_rank_local').head(10)


,track_name,artist_names,cidade,melhor_rank_local,melhor_rank_eixo
2746,Tem Café,Henry Freitas,Salvador,1,52
2743,Tem Café,Henry Freitas,Fortaleza,1,52
1598,Macetando,Ivete Sangalo,Salvador,1,51
915,Duas,Nadson O Ferinha,Salvador,1,101
2745,Tem Café,Henry Freitas,Recife,1,52
208,Americana Na Vaquejada,Grandão Vaqueiro,Fortaleza,2,69
210,Americana Na Vaquejada,Grandão Vaqueiro,Recife,2,69
626,Chorei na Vaquejada,Eric Land,Recife,2,74
627,Chorei na Vaquejada,Eric Land,Salvador,2,74
796,Deixa eu botar meu boneco - Baile do Polly 2.0,Oh Polêmico,Salvador,3,65


In [17]:
import pandas as pd

# 1. Filtrar com foco exclusivo no eixo tradicional e nas capitais de nicho (Manaus e BH)
df_eixo = df_garimpo[df_garimpo['cidade'].isin(['São Paulo', 'Rio de Janeiro'])]
df_resistencia = df_garimpo[df_garimpo['cidade'].isin(['Manaus', 'Belo Horizonte'])]

# 2. Descobrir a melhor posição de cada música no Eixo RJ-SP
pico_no_eixo = df_eixo.groupby(['track_name', 'artist_names'])['rank'].min().reset_index()
pico_no_eixo.rename(columns={'rank': 'melhor_rank_eixo'}, inplace=True)

# 3. Descobrir a melhor posição de cada música em Belém ou BH
pico_na_resistencia = df_resistencia.groupby(['track_name', 'artist_names', 'cidade'])['rank'].min().reset_index()
pico_na_resistencia.rename(columns={'rank': 'melhor_rank_local'}, inplace=True)

# 4. Cruzar os dados (LEFT JOIN)
analise_monopolio = pd.merge(pico_na_resistencia, pico_no_eixo, on=['track_name', 'artist_names'], how='left')
analise_monopolio['melhor_rank_eixo'] = analise_monopolio['melhor_rank_eixo'].fillna(101)

# 5. O FILTRO: Estourou no Top 10 local, mas passou longe do Sudeste (pior que #85 ou nem entrou)
filtro_garimpo_real = (analise_monopolio['melhor_rank_local'] <= 10) & (analise_monopolio['melhor_rank_eixo'] >= 85)
resultado_refinado = analise_monopolio[filtro_garimpo_real]

# 6. Exibimos as joias escondidas
print("GARIMPO INDIE REFINADO (Fenômenos locais isolados do mainstream do Sudeste):")
resultado_refinado.sort_values(by='melhor_rank_local').head(15)


GARIMPO INDIE REFINADO (Fenômenos locais isolados do mainstream do Sudeste):


,track_name,artist_names,cidade,melhor_rank_local,melhor_rank_eixo


In [18]:
import pandas as pd

# 1. Filtrar com foco exclusivo no eixo tradicional e nas capitais de nicho (Recife e BH)
df_eixo = df_garimpo[df_garimpo['cidade'].isin(['São Paulo', 'Rio de Janeiro'])]
df_resistencia = df_garimpo[df_garimpo['cidade'].isin(['Recife', 'Belo Horizonte'])]

# 2. Descobrir a melhor posição de cada música no Eixo RJ-SP
pico_no_eixo = df_eixo.groupby(['track_name', 'artist_names'])['rank'].min().reset_index()
pico_no_eixo.rename(columns={'rank': 'melhor_rank_eixo'}, inplace=True)

# 3. Descobrir a melhor posição de cada música em Belém ou BH
pico_na_resistencia = df_resistencia.groupby(['track_name', 'artist_names', 'cidade'])['rank'].min().reset_index()
pico_na_resistencia.rename(columns={'rank': 'melhor_rank_local'}, inplace=True)

# 4. Cruzar os dados (LEFT JOIN)
analise_monopolio = pd.merge(pico_na_resistencia, pico_no_eixo, on=['track_name', 'artist_names'], how='left')
analise_monopolio['melhor_rank_eixo'] = analise_monopolio['melhor_rank_eixo'].fillna(101)

# 5. O FILTRO: Estourou no Top 10 local, mas passou longe do Sudeste (pior que #85 ou nem entrou)
filtro_garimpo_real = (analise_monopolio['melhor_rank_local'] <= 10) & (analise_monopolio['melhor_rank_eixo'] >= 85)
resultado_refinado = analise_monopolio[filtro_garimpo_real]

# 6. Exibimos as joias escondidas
print("GARIMPO INDIE REFINADO (Fenômenos locais isolados do mainstream do Sudeste):")
resultado_refinado.sort_values(by='melhor_rank_local').head(15)


GARIMPO INDIE REFINADO (Fenômenos locais isolados do mainstream do Sudeste):


,track_name,artist_names,cidade,melhor_rank_local,melhor_rank_eixo
313,Coisas Que Eu Sei,Felipe Amorim,Recife,4,89
37,A Gente Se Entrega,NATTAN,Recife,7,101
255,Cadê Seu Namorado Moça?,Thales Lessa,Recife,9,101
442,Duas,Nadson O Ferinha,Recife,10,101
